# Multilingual Image Captioning: train on a Colab GPU

Runtime > Change runtime type > **T4 GPU**. This notebook clones the repo, prepares an MS-COCO 2014 subset, caches CNN features, trains InceptionV3 and ResNet50 runs at the same settings, evaluates them and zips the checkpoint for a GitHub Release.

Rough timings on a T4 with 6,000 images: download ~5 min, features ~3 min per backbone, training ~1-2 min per epoch.

In [1]:
!nvidia-smi -L
!git clone https://github.com/Anudeepthi905/multilingual-image-captioning.git
%cd multilingual-image-captioning
# Keep Colab's own TensorFlow/NumPy (built for its CUDA); install everything else.
!grep -vE '^(tensorflow|tensorflow-cpu|keras|numpy)==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt pytest
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

GPU 0: Tesla T4 (UUID: GPU-724e3f37-ba1e-5203-b989-30c01cfd529d)
Cloning into 'multilingual-image-captioning'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 118 (delta 40), reused 115 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 280.54 KiB | 1.52 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/multilingual-image-captioning
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.3 which is incompatible.
dataproc-spark-connect 1.1.0 requires tqdm>=4.67, but you have tqdm 4.66.5 which is incompatible.
google-cloud-language 2.2

VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow/core/framework/attr_value.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [ ]:
# Settings: raise NUM_IMAGES (e.g. 30000) if you have time; add --full-zip for > ~20k images.
NUM_IMAGES = 6000
VOCAB = 5000
EPOCHS = 20
!python -m pytest -q
!python -m src.data --num-images {NUM_IMAGES}

In [ ]:
# Cache CNN features once per backbone (training never re-runs the CNN).
!python -m src.features --backbone inceptionv3 --num-images {NUM_IMAGES} --batch-size 64
!python -m src.features --backbone resnet50 --num-images {NUM_IMAGES} --batch-size 64

In [ ]:
!python -m src.train --backbone inceptionv3 --vocab-size {VOCAB} --num-images {NUM_IMAGES} --epochs {EPOCHS}
!python -m src.train --backbone resnet50 --vocab-size {VOCAB} --num-images {NUM_IMAGES} --epochs {EPOCHS}
# Optional: larger vocabulary
# !python -m src.train --backbone inceptionv3 --vocab-size 7000 --num-images {NUM_IMAGES} --epochs {EPOCHS}

In [ ]:
INC = f'inceptionv3_v{VOCAB}_n{NUM_IMAGES}_e{EPOCHS}'
RES = f'resnet50_v{VOCAB}_n{NUM_IMAGES}_e{EPOCHS}'
for run in [INC, RES]:
    !python -m src.evaluate --run {run}
    !python -m src.evaluate --run {run} --beam 3
import pandas as pd; pd.read_csv('results/metrics.csv')

In [ ]:
from IPython.display import Image, display
!python -m src.caption --image assets/samples/$(ls assets/samples | grep -m1 jpg) --run {INC} --beam 3 --plot
for f in ['results/comparison.png', f'results/loss_{INC}.png', f'results/loss_{RES}.png']:
    display(Image(f))

In [ ]:
# Publish the best run for the demo: export weights only and upload to a Hugging Face
# model repo (log in with a write token first). The Space reads MIC_HF_MODEL_REPO.
HF_REPO = 'your-hf-username/multilingual-image-captioning'  # <- change this
from huggingface_hub import notebook_login; notebook_login()
!python -m src.export --run {INC} --push {HF_REPO}
# Download the results folder to commit metrics.csv and the plots.
!zip -r results.zip results README.md
from google.colab import files
files.download('results.zip')